# 01 — Exploratory Data Analysis
Load raw FRED data, inspect stationarity, seasonality, and correlations between leading indicators and targets.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from statsmodels.tsa.stattools import adfuller
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

plt.rcParams['figure.figsize'] = (12, 4)
sns.set_theme(style='whitegrid')
print('Setup complete')

In [ ]:
# Load raw data (run data_loader.py first if this file doesn't exist)
df = pd.read_csv('../data/raw/macro_raw.csv', index_col='date', parse_dates=True)
print(df.shape)
df.tail()

## 1. Time series plots

In [ ]:
targets = ['gdp', 'cpi', 'unemployment']
indicators = [c for c in df.columns if c not in targets]

fig, axes = plt.subplots(len(targets), 1, figsize=(14, 10))
for ax, col in zip(axes, targets):
    s = df[col].dropna()
    ax.plot(s, linewidth=1.2)
    ax.set_title(col.upper(), fontsize=11)
    ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('../outputs/figures/targets_raw.png', dpi=150)
plt.show()

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(16, 10))
for ax, col in zip(axes.flatten(), indicators):
    s = df[col].dropna()
    ax.plot(s, linewidth=0.8)
    ax.set_title(col, fontsize=9)
    ax.grid(True, alpha=0.3)
plt.suptitle('Leading Indicators (raw levels)', fontsize=12, y=1.01)
plt.tight_layout()
plt.savefig('../outputs/figures/indicators_raw.png', dpi=150)
plt.show()

## 2. Stationarity test (ADF)

In [ ]:
def adf_test(series, name):
    s = series.dropna()
    result = adfuller(s, autolag='AIC')
    return {
        'series': name,
        'adf_stat': round(result[0], 3),
        'p_value': round(result[1], 4),
        'stationary': result[1] < 0.05
    }

adf_results = [adf_test(df[c], c) for c in df.columns if df[c].dropna().shape[0] > 30]
adf_df = pd.DataFrame(adf_results)
print(adf_df.to_string(index=False))
# Non-stationary series need differencing/pct_change in features.py

## 3. Correlation heatmap (monthly resampled)

In [ ]:
from src.features import resample_to_monthly, make_stationary

df_monthly = resample_to_monthly(df)
df_stationary = make_stationary(df_monthly).dropna()

fig, ax = plt.subplots(figsize=(12, 10))
corr = df_stationary.corr()
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt='.2f', cmap='RdBu_r',
            center=0, ax=ax, square=True, linewidths=0.5)
ax.set_title('Correlation Matrix (stationary, monthly)', fontsize=12)
plt.tight_layout()
plt.savefig('../outputs/figures/correlation_heatmap.png', dpi=150)
plt.show()

## 4. Lead-lag analysis: do indicators lead GDP?

In [ ]:
gdp_pct = df_stationary['gdp'].dropna()

fig, axes = plt.subplots(3, 3, figsize=(16, 10))
for ax, col in zip(axes.flatten(), indicators):
    ind = df_stationary[col].dropna()
    combined = pd.concat([gdp_pct, ind], axis=1).dropna()
    lags = range(-6, 7)  # -6 = indicator leads GDP by 6m, +6 = lags
    cors = [combined[col].shift(lag).corr(combined['gdp']) for lag in lags]
    ax.bar(lags, cors, color=['steelblue' if c > 0 else 'tomato' for c in cors])
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title(f'{col}', fontsize=9)
    ax.set_xlabel('Lag (months)')
    ax.set_ylabel('Corr with GDP')
    ax.grid(True, alpha=0.3)
plt.suptitle('Cross-correlation: indicators vs GDP (negative lag = indicator leads)', fontsize=11)
plt.tight_layout()
plt.savefig('../outputs/figures/leadlag_gdp.png', dpi=150)
plt.show()

## 5. ACF / PACF of target series

In [ ]:
for col in targets:
    s = df_stationary[col].dropna()
    if len(s) < 20:
        continue
    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 3))
    plot_acf(s, lags=24, ax=ax1, title=f'ACF — {col}')
    plot_pacf(s, lags=24, ax=ax2, title=f'PACF — {col}', method='ywm')
    plt.tight_layout()
    plt.savefig(f'../outputs/figures/acf_{col}.png', dpi=150)
    plt.show()
    # Use PACF cutoff to guide ARIMA p parameter in baseline.py